# Diagnostic Analytics — BMI Category, Smoker Status, Age, and Cost Outliers

**Goal:** Go beyond simple averages to understand *why* charges vary — combining BMI category with smoker status (the strongest cost driver in this dataset), checking correlations with age, and profiling who the high-cost outliers actually are.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('data/processed/insurance_clean.csv')
print(df.shape)

## 1. BMI Categories

Splitting BMI into standard clinical categories.

In [ ]:
df['bmi_category'] = pd.cut(
    df['bmi'],
    bins=[0, 18.5, 25, 30, float('inf')],
    labels=['Underweight', 'Normal', 'Overweight', 'Obese']
)
print(df['bmi_category'].value_counts())

**Finding:** Most patients fall into the Obese (704) and Overweight (386) categories, with far fewer Underweight patients (21).

In [ ]:
bmi_only = df.groupby('bmi_category', observed=True)['charges'].agg(['mean', 'median', 'count']).round(2)
print("=== BMI Category Only ===")
print(bmi_only)

**Finding:** Average charges rise with BMI category — Obese patients average **$15,580.70**, noticeably higher than Normal ($10,435.44) or Underweight ($8,657.62). But this alone doesn't tell the full story, as we'll see below.

## 2. Smoker Status

In [ ]:
smoker_only = df.groupby('smoker')['charges'].agg(['mean', 'median', 'count']).round(2)
print("=== Smoker Status Only ===")
print(smoker_only)

**Finding:** Smokers average **$32,050.23** in charges versus **$8,440.66** for non-smokers — roughly **3.8x higher**. This is by far the single strongest driver of cost seen so far, even stronger than BMI category alone.

## 3. BMI Category x Smoker Status (Key Combination)

This is the most important cut in the analysis — combining the two factors reveals the real cost driver.

In [ ]:
combo = df.groupby(['bmi_category', 'smoker'], observed=True)['charges'].agg(['mean', 'median', 'count']).round(2)
print("=== BMI Category x Smoker ===")
print(combo)

**Finding — the clearest insight in this dataset:** Obese smokers average **$41,692.81** in charges, compared to **$8,866.16** for obese non-smokers — a **4.70x** difference. Smoking status changes the cost picture far more dramatically than BMI category does on its own. Being obese barely raises costs for non-smokers, but combined with smoking, it's the single highest-cost segment in the dataset.

## 4. Correlation: Age, BMI, and Charges

In [ ]:
correlation = df[['age', 'bmi', 'charges']].corr()
print(correlation.round(3))

**Finding:** Both `age` (0.298) and `bmi` (0.198) show only weak positive correlation with `charges` on their own. This confirms that neither age nor BMI alone explains cost well — smoking status is the dominant factor, and age/BMI only matter substantially when combined with it.

## 5. Age Groups

In [ ]:
df['age_group'] = pd.cut(df['age'], bins=[17, 29, 39, 49, 59, 100],
                          labels=['18-29', '30-39', '40-49', '50-59', '60+'])

age_analysis = df.groupby('age_group', observed=True)[['bmi', 'charges']].mean().round(2)
print(age_analysis)

**Finding:** Average charges rise steadily with age, from $9,200.62 (18-29) to $21,248.02 (60+) — more than doubling. Average BMI also creeps up slightly with age, but far less dramatically than charges do.

## 6. Outlier Detection (IQR Method)

In [ ]:
Q1 = df['charges'].quantile(0.25)
Q3 = df['charges'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Q1: {Q1:.2f}, Q3: {Q3:.2f}, IQR: {IQR:.2f}")
print(f"Lower bound: {lower_bound:.2f}, Upper bound: {upper_bound:.2f}")

outliers = df[(df['charges'] < lower_bound) | (df['charges'] > upper_bound)]
print(f"\nNumber of outliers: {len(outliers)}")

**Finding:** 139 patients (about 10.4% of the dataset) fall outside the IQR bounds on the high-charges side.

In [ ]:
print("Smoker distribution among outliers:")
print(outliers['smoker'].value_counts(normalize=True).round(3))

print("\nBMI category distribution among outliers:")
print(outliers['bmi_category'].value_counts(normalize=True).round(3))

**Finding — who the outliers actually are:** Among the 139 high-cost outliers, **97.8% are smokers** and **96.4% are classified as Obese**. These are not random data errors — they are a clearly identifiable, real segment of the population (obese smokers), which directly supports the business question of who drives the highest costs.

## 7. Visualization — Charges Distribution by Smoker Status

In [ ]:
sns.set_style('whitegrid')
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='smoker', y='charges', hue='smoker',
            palette={'yes': '#e07a5f', 'no': '#3d9970'}, legend=False)
plt.title('Charges Distribution by Smoker Status (Outliers Visible)', fontsize=13, fontweight='bold')
plt.xlabel('Smoker')
plt.ylabel('Charges ($)')
plt.tight_layout()
plt.savefig('reports/outliers_boxplot_smoker.png', dpi=150)
plt.show()

**Finding:** The boxplot makes the pattern immediately visible — smokers (left) have a much higher and wider charges distribution, while nearly all statistical outliers (dots) appear on the non-smoker side, representing non-smokers with unusually high costs (likely tied to high BMI or other factors).

## 8. Visualization — Charges Distribution by BMI Category

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='bmi_category', y='charges',
            order=['Underweight', 'Normal', 'Overweight', 'Obese'],
            color='#4a7c8c')
plt.title('Charges Distribution by BMI Category (Outliers Visible)', fontsize=13, fontweight='bold')
plt.xlabel('BMI Category')
plt.ylabel('Charges ($)')
plt.tight_layout()
plt.savefig('reports/outliers_boxplot_bmi.png', dpi=150)
plt.show()

## Summary

| Finding | Result |
|---|---|
| Obese smokers vs. obese non-smokers | **4.70x** higher average charges ($41,692.81 vs. $8,866.16) |
| Smokers vs. non-smokers (overall) | **3.8x** higher average charges ($32,050.23 vs. $8,440.66) |
| Age/BMI correlation with charges | Weak on their own (0.298 and 0.198) — smoking dominates |
| Charges by age group | More than double from 18-29 ($9,200.62) to 60+ ($21,248.02) |
| High-cost outliers (IQR) | 139 patients (10.4%) — **97.8% smokers, 96.4% obese** |

**Bottom line:** Smoking status — especially combined with obesity — is the dominant driver of medical costs in this dataset, far outweighing age or BMI alone.